# 02. Baseline Linear Modeling & Hyperparameter Search
> **Benchmarking regularized Logistic Regression vs. Naive Baseline**

Part of the **GreenlightML** open-source film risk modeling and portfolio optimization engine.


> **Draft / not validated.** Scaffold only. Raw rows are split before any learned preprocessing (fit on train only), but the target proxy, population, prediction date, and split policy are **not frozen** yet, so no score here is accepted evidence. See `docs/METHODOLOGY.md`.

## 1. Environment & Feature Preparation
Load clean data and extract pre-release feature matrices.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier

sys.path.append(os.path.abspath('..'))
from src.data import load_and_merge_data, filter_reported_financials, split_population
from src.features import FeatureEncoder
from src.models import get_logistic_regression
from src.metrics import evaluate_model

df_reported = filter_reported_financials(load_and_merge_data())

# Split the raw rows FIRST; every learned step is fit on train only.
# The split policy (random vs. chronological vs. group-aware) is a result-defining
# choice and is NOT frozen: it is named explicitly here instead of being hardcoded,
# so this scaffold cannot pick it silently. See docs/METHODOLOGY.md.
train_df, test_df = split_population(df_reported, policy="random_stratified_provisional")
encoder = FeatureEncoder(max_tfidf_features=100).fit(train_df)
X_train = encoder.transform(train_df)
X_test = encoder.transform(test_df)
y_train, y_test = train_df['target_nominal'], test_df['target_nominal']
test_slice = test_df

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

## 2. Benchmark: Naive Dummy Classifier
Demonstrate why single-point accuracy is misleading on skewed classes.


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_sc, y_train)
dummy_pred = dummy.predict(X_test_sc)

res_dummy = evaluate_model("Naive Dummy Baseline", "Passive", y_test, dummy_pred, None, test_slice)
pd.DataFrame([res_dummy])


## 3. Model 1: Logistic Regression & Systematic Hyperparameter Tuning
Vary regularization strength C and class weighting using Stratified 5-Fold Cross-Validation.


In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(get_logistic_regression(), param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train_sc, y_train)

print(f"Best CV ROC-AUC: {grid.best_score_:.4f}")
print(f"Best Hyperparameters: {grid.best_params_}")


## 4. Out-of-Sample Holdout Evaluation
Evaluate tuned Logistic Regression against unseen test titles.


In [ ]:
best_lr = grid.best_estimator_
y_pred_lr = best_lr.predict(X_test_sc)
y_prob_lr = best_lr.predict_proba(X_test_sc)[:, 1]

res_lr = evaluate_model("Logistic Regression (Tuned)", "Pre-release", y_test, y_pred_lr, y_prob_lr, test_slice)
pd.DataFrame([res_dummy, res_lr])
